In [5]:
# IMPORT LIBRARIES

import os
import re
import string
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
# DEFINE PROJECT PATHS

BASE_DIR = r"C:\Users\acer\Documents\Clinical_Trial_Classification"

INPUT_PATH = os.path.join(
    BASE_DIR, "outputs", "cleaned_clinical_trials.csv"
)

OUTPUT_PATH = os.path.join(
    BASE_DIR, "outputs", "preprocessed_clinical_trials.csv"
)

print("Input file exists:", os.path.exists(INPUT_PATH))
print("Output folder exists:", os.path.isdir(os.path.join(BASE_DIR, "outputs")))

Input file exists: True
Output folder exists: True


In [7]:
# LOAD CLEANED DATASET

df = pd.read_csv(INPUT_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("\nRequired columns:")
print(df[["brief_summary", "source_condition_query"]].head())

Dataset loaded successfully.
Shape: (60337, 16)

Required columns:
                                       brief_summary source_condition_query
0  Breast cancer patients often have perioperativ...          breast cancer
1  Many breast cancer patients experience psychol...          breast cancer
2  Based on an American study by Scherer et al., ...          breast cancer
3  Compare the effect of ropivacaine versus place...          breast cancer
4  Phase 1 dose escalation and expansion study of...          breast cancer


In [10]:
# VALIDATE REQUIRED COLUMNS

required_columns = ["brief_summary", "source_condition_query"]

print("Required columns present:",
      all(col in df.columns for col in required_columns))

print("\nMissing values:")
print(df[required_columns].isnull().sum())

print("\nEmpty text summaries:",
      df["brief_summary"].astype(str).str.strip().eq("").sum())

Required columns present: True

Missing values:
brief_summary             0
source_condition_query    0
dtype: int64

Empty text summaries: 0


In [11]:
# CLEAN TEXT

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"_+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["cleaned_summary"] = df["brief_summary"].apply(clean_text)

print("Text cleaning completed.")
print(df[["brief_summary", "cleaned_summary"]].head(3))

Text cleaning completed.
                                       brief_summary  \
0  Breast cancer patients often have perioperativ...   
1  Many breast cancer patients experience psychol...   
2  Based on an American study by Scherer et al., ...   

                                     cleaned_summary  
0  breast cancer patients often have perioperativ...  
1  many breast cancer patients experience psychol...  
2  based on an american study by scherer et al it...  


In [12]:
# SET UP NLTK RESOURCES

import nltk

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng")

print("NLTK resource setup completed.")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


NLTK resource setup completed.


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [13]:
# IMPORT NLP TOOLS

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

print("NLP tools imported successfully.")

NLP tools imported successfully.


In [14]:
# INITIALIZE NLP TOOLS

stop_words = set(stopwords.words("english"))
stop_words -= {"no", "not", "nor"}

lemmatizer = WordNetLemmatizer()

print("English stopwords loaded:", len(stop_words))
print("Lemmatizer initialized successfully.")

English stopwords loaded: 195
Lemmatizer initialized successfully.


In [15]:
# MAP POS TAGS FOR LEMMATIZATION

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

print("POS mapping function defined successfully.")

POS mapping function defined successfully.


In [16]:
# NLP PREPROCESSING FUNCTION

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tagged_tokens = nltk.pos_tag(tokens)

    processed_tokens = []

    for word, tag in tagged_tokens:
        if word.isalpha() and word not in stop_words:
            pos = get_wordnet_pos(tag)
            lemma = lemmatizer.lemmatize(word, pos)
            processed_tokens.append(lemma)

    return " ".join(processed_tokens)

print("NLP preprocessing function defined successfully.")

NLP preprocessing function defined successfully.


In [17]:
# TEST NLP PREPROCESSING

sample_text = "The patients were not recovering quickly. The diseases were treated with new therapies."

print("Original text:")
print(sample_text)

print("\nProcessed text:")
print(preprocess_text(sample_text))

Original text:
The patients were not recovering quickly. The diseases were treated with new therapies.

Processed text:
patient not recover quickly disease treat new therapy


In [18]:
# APPLY NLP PREPROCESSING

df["processed_summary"] = df["cleaned_summary"].apply(preprocess_text)

print("NLP preprocessing completed.")
print("Dataset shape:", df.shape)
print(df[["cleaned_summary", "processed_summary"]].head(3))

NLP preprocessing completed.
Dataset shape: (60337, 18)
                                     cleaned_summary  \
0  breast cancer patients often have perioperativ...   
1  many breast cancer patients experience psychol...   
2  based on an american study by scherer et al it...   

                                   processed_summary  
0  breast cancer patient often perioperative emot...  
1  many breast cancer patient experience psycholo...  
2  base american study scherer et al hypothesize ...  


In [19]:
# VALIDATE PREPROCESSED TEXT

print("Missing processed summaries:",
      df["processed_summary"].isna().sum())

print("Empty processed summaries:",
      df["processed_summary"].str.strip().eq("").sum())

print("Number of target categories:",
      df["source_condition_query"].nunique())

print("\nProcessed summary length statistics:")
print(df["processed_summary"].str.split().str.len().describe())

Missing processed summaries: 0
Empty processed summaries: 0
Number of target categories: 8

Processed summary length statistics:
count    60337.000000
mean        64.218821
std         58.375037
min          2.000000
25%         24.000000
50%         44.000000
75%         85.000000
max        546.000000
Name: processed_summary, dtype: float64


In [20]:
# SAVE PREPROCESSED DATASET

df.to_csv(OUTPUT_PATH, index=False)

print("Preprocessed dataset saved successfully.")
print("Output file:", OUTPUT_PATH)
print("Final shape:", df.shape)

Preprocessed dataset saved successfully.
Output file: C:\Users\acer\Documents\Clinical_Trial_Classification\outputs\preprocessed_clinical_trials.csv
Final shape: (60337, 18)
